In [1]:
import pandas as pd

In [5]:
df = pd.read_csv("card_transdata.csv")
df.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


In [40]:
X = df.drop(columns="fraud")
y = df["fraud"].astype(int)

In [42]:
from sklearn.model_selection import train_test_split
#splits the entire data set into 80% train, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
#splits the 80% train into 75% train, 25% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42,
    stratify=y_train_val
)

In [44]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(max_iter=1000, random_state=42))])

In [15]:
from sklearn.metrics import confusion_matrix, classification_report

val_prob = model.predict_proba(X_val)[:, 1]
val_pred_050 = (val_prob >= 0.50).astype(int)

print(confusion_matrix(y_val, val_pred_050))
print(classification_report(y_val, val_pred_050, digits=3))

[[181185   1335]
 [  7017  10463]]
              precision    recall  f1-score   support

           0      0.963     0.993     0.977    182520
           1      0.887     0.599     0.715     17480

    accuracy                          0.958    200000
   macro avg      0.925     0.796     0.846    200000
weighted avg      0.956     0.958     0.955    200000



In [33]:
from sklearn.metrics import confusion_matrix, classification_report

val_prob = model.predict_proba(X_val)[:, 1]
val_pred_050 = (val_prob >= 0.40).astype(int)

print(confusion_matrix(y_val, val_pred_050))
print(classification_report(y_val, val_pred_050, digits=3))

[[180674   1846]
 [  5459  12021]]
              precision    recall  f1-score   support

           0      0.971     0.990     0.980    182520
           1      0.867     0.688     0.767     17480

    accuracy                          0.963    200000
   macro avg      0.919     0.839     0.874    200000
weighted avg      0.962     0.963     0.962    200000



In [146]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, average_precision_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

# Baseline predictions
baseline_pred = baseline.predict(X_val)

# Logistic regression predictions at threshold 0.50
model_prob = model.predict_proba(X_val)[:, 1]
model_pred = (model_prob >= 0.38).astype(int)

print("Baseline:")
print(classification_report(y_val, baseline_pred, digits=3, zero_division=0))

print("Logistic regression:")
print(classification_report(y_val, model_pred, digits=3, zero_division=0))
print("Average precision:", average_precision_score(y_val, model_prob))

Baseline:
              precision    recall  f1-score   support

           0      0.913     1.000     0.954    182520
           1      0.000     0.000     0.000     17480

    accuracy                          0.913    200000
   macro avg      0.456     0.500     0.477    200000
weighted avg      0.833     0.913     0.871    200000

Logistic regression:
              precision    recall  f1-score   support

           0      0.972     0.989     0.980    182520
           1      0.858     0.705     0.774     17480

    accuracy                          0.964    200000
   macro avg      0.915     0.847     0.877    200000
weighted avg      0.962     0.964     0.962    200000

Average precision: 0.7979488660151568


In [150]:
from sklearn.metrics import precision_score, recall_score
best_threshold = None
min_recall = 0.7
best_precision = -1
best_recall = 0

for i in range(1, 1000):
    threshold = i / 1000
    predictions = (val_prob >= threshold).astype(int)
    
    precision = precision_score(y_val, predictions, zero_division = 0)
    recall = recall_score(y_val, predictions)

    if recall >= min_recall and precision > 0.85:
        best_precision = precision
        best_threshold = threshold
        best_recall = recall

print(best_threshold)
print(best_precision)
print(best_recall)

0.386
0.8606090442365848
0.7000572082379862
